# 04. Baseline Evaluation and Normalization

- Parent issue: `#19`
- Status: `active`
- Summary: Define and validate the repeatable baseline evaluation contract so that improvements in later waves can be measured without self-deception.

## Audience and Why It Matters

Evaluation owners, reviewers, and anyone comparing model changes across waves.

This notebook defines the **normalization contract** - the versioned rules that map raw model output to a comparable answer string - and validates the full evaluation pipeline (`ingest -> predict -> normalize -> score -> report`) end-to-end. It is the gate that later training and prompting notebooks (issues #21 onward) must satisfy before their results can be trusted.

## Decision / Hypothesis

**Default scoring contract:** strip leading/trailing whitespace (`strip_v1`) before strict exact-match comparison.

Rationale (from `docs/analysis/ADVERSARIAL_REVIEW.md` and `docs/architecture/ARCHITECTURE.md`):
- The competition benchmark uses plain string answers (binary strings, cipher text, numeric results) - not `\\boxed{}` LaTeX.
- Exact-match is the harshest and most auditable contract; permissive variants must earn their justification.
- Normalizer versioning ensures that any change to the scoring rule is an explicit artifact change, not silent code drift.

**Provisional assumption** (to be confirmed against the official Kaggle evaluator):  
The competition uses exact-match on the model's raw output. If official rules confirm character-exact scoring, downgrade `NORMALIZER_ID` to `exact_v1`. Either way, re-inference is not needed since raw predictions are preserved in `predictions.jsonl`.

## Environment and Reproduction

- Python: 3.12 (Kaggle hosted runtime)
- GPU required — this notebook runs the real Nemotron-3-Nano-30B model on Kaggle.
- Run on Kaggle: attach the `metric/nemotron-3-nano-30b-a3b-bf16` model and the eval splits dataset before executing.
- Artifacts are written to `data/eval/runs/<run_id>/` (git-ignored via `data/**`).
- **Reloadable:** run all cells in order from a fresh kernel. No prior notebook state is required.

In [ ]:
import sys
import hashlib
import platform
from dataclasses import asdict, fields as dc_fields
from pathlib import Path

# Robust repo-root detection: walk up until we find src/ + notebooks/
_cwd = Path.cwd()
REPO_ROOT = _cwd
for _p in [_cwd] + list(_cwd.parents):
    if (_p / "src").is_dir() and (_p / "notebooks").is_dir():
        REPO_ROOT = _p
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.platform()}")

# Smoke-test key evaluation modules
from src.contracts import EvalRecord
from src.evaluation.normalization import (
    available_normalizers, normalize, BUILTIN_NORMALIZERS,
)
from src.evaluation.config import make_run_config
from src.evaluation.runner import (
    PredictRequest, PredictResponse, run_baseline_eval,
)
from src.evaluation.reporting import (
    read_eval_records_jsonl, read_summary_json,
)
from src.evaluation.splits import SplitArtifactRow, read_split_jsonl
from src.evaluation.scoring import score_records
from src.evaluation.golden_gate import evaluate_golden_gate, summarize_gate

print(f"\nRegistered normalizers : {available_normalizers()}")
print("All evaluation modules imported OK.")


## 1. Contract Documentation

### EvalRecord Shape

Every evaluated prediction produces one `EvalRecord` (defined in `src/contracts.py`). The record is flat for JSONL serialization and self-contained for replay. Key attribution fields:

| Field | Purpose |
|---|---|
| `run_id` | Unique run identifier stamped on every record |
| `example_id` | Links to the canonical example in the split artifact |
| `model_id` | Exact model checkpoint used |
| `prompt_template_id` | Prompt format applied |
| `normalizer_id` | Which rule produced `normalized_prediction` |
| `decode_config` | Snapshot of generation hyperparameters |

### Normalization Policy

| `normalizer_id` | Transformation | When to use |
|---|---|---|
| `exact_v1` | Identity (no transformation) | When competition scoring is truly character-exact |
| `strip_v1` | `str.strip()` only | **Default baseline** - safe for output with trailing newlines or spaces |
| `collapse_ws_v1` | Collapse internal whitespace | When answer text has variable internal spacing |
| `last_line_v1` | Last non-empty line, stripped | When output format is `<think>...</think>\nfinal_answer` |

**Immutability rule:** once a `normalizer_id` is published its observable behavior is frozen. To change parsing logic, register a new version id (e.g., `strip_v2`). This prevents silent history rewriting on committed eval records.

In [ ]:
print("EvalRecord fields:")
print(f"  {'field':<25} type")
print(f"  {'-'*25} {'-'*35}")
for f in dc_fields(EvalRecord):
    print(f"  {f.name:<25} {f.type}")


In [ ]:
demo_inputs = [
    ("42",                           None,             "clean answer"),
    ("  42  ",                       None,             "leading/trailing spaces"),
    ("hello  world",                 None,             "multiple internal spaces"),
    ("<think>reasoning</think>\n42", None,             "reasoning + answer"),
    ("FF",                           "numeral_system", "correct hex"),
    ("ff",                           "numeral_system", "lowercase hex (case drift)"),
    ("",                             None,             "blank prediction"),
]

print(f"{'raw':<40} {'normalizer_id':<22} {'normalized'}")
print("-" * 85)
for raw, cat, desc in demo_inputs:
    first = True
    for nid in available_normalizers():
        result = normalize(raw, normalizer_id=nid, category=cat)
        label = f"  # {desc}" if first else ""
        print(f"{repr(raw):<40} {nid:<22} {repr(result)}{label}")
        first = False
    print()


## 2. Run Configuration

Every eval run is described by an `EvalRunConfig`. The config is persisted as `run_config.json` alongside the records. Every `EvalRecord` carries copies of `run_id`, `model_id`, `prompt_template_id`, `normalizer_id`, `seed`, and `decode_config` so a single record can be re-evaluated without the original config file.

**Provisional decode config:**
- `temperature=0.0`, `do_sample=False`: deterministic greedy decoding for the exact-match baseline.
- `enable_thinking=False`: no extended thinking; issue #21 will sweep this.
- `max_new_tokens=512`: matches the expected short-answer format of the competition.

In [ ]:
# ── Run identity ──────────────────────────────────────────────────────────────
RUN_ID             = "baseline-real-v1"
MODEL_ID           = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
PROMPT_TEMPLATE_ID = "zero-shot-v1"
NORMALIZER_ID      = "strip_v1"   # strip only; swap to last_line_v1 if model wraps in <think>
DATASET_VERSION    = "kaggle-v1"
SEED               = 42

DECODE_CONFIG = {
    "temperature": 0.0,
    "top_p": 1.0,
    "max_new_tokens": 512,
    "do_sample": False,
    "enable_thinking": False,
}

run_config = make_run_config(
    run_id=RUN_ID,
    model_id=MODEL_ID,
    prompt_template_id=PROMPT_TEMPLATE_ID,
    normalizer_id=NORMALIZER_ID,
    seed=SEED,
    split="val",
    dataset_version=DATASET_VERSION,
    decode_config=DECODE_CONFIG,
)

print("Run config:")
for k, v in asdict(run_config).items():
    print(f"  {k:<22}: {v!r}")

## 3. Data Ingest

**Provisional assumption:** The `SplitArtifactRow` artifacts from issue #18 (`data/eval/val.jsonl` and `data/eval/golden.jsonl`) are not yet available because notebooks 02 and 03 are still scaffolded.

This cell attempts to load those files if they exist, then falls back to synthetic rows covering all known competition categories. **Replace** this cell when #17 and #18 produce real split artifacts - the pipeline code below is unchanged.

In [ ]:
VAL_ARTIFACT_PATH    = REPO_ROOT / "data" / "eval" / "val.jsonl"
GOLDEN_ARTIFACT_PATH = REPO_ROOT / "data" / "eval" / "golden.jsonl"

# Synthetic val examples - 2-3 per competition category
_SYNTHETIC_VAL = [
    # (example_id, category, prompt, gold)
    ("synth-bit-001",    "bit_manipulation",
     "XOR key='1010'. f('0000')='1010',f('1111')='0101'. f('1100')=?",    "0110"),
    ("synth-bit-002",    "bit_manipulation",
     "XOR key='1010'. f('0000')='1010',f('1111')='0101'. f('0011')=?",    "1001"),
    ("synth-bit-003",    "bit_manipulation",
     "XOR key='1010'. f('0000')='1010',f('1111')='0101'. f('1111')=?",    "0101"),
    ("synth-unit-001",   "unit_conversion",
     "Convert 100 km/h to m/s. Round to 2 decimal places.",                             "27.78"),
    ("synth-unit-002",   "unit_conversion",
     "Convert 1 hour to seconds.",                                                       "3600"),
    ("synth-unit-003",   "unit_conversion",
     "Convert 1 km to meters.",                                                          "1000"),
    ("synth-cipher-001", "text_cipher",
     "Decode ROT-13: 'uryyb'",                                                         "hello"),
    ("synth-cipher-002", "text_cipher",
     "Decode ROT-13: 'jbeyq'",                                                         "world"),
    ("synth-cipher-003", "text_cipher",
     "Decode ROT-13: 'clguba'",                                                        "python"),
    ("synth-num-001",    "numeral_system",
     "Convert decimal 255 to hexadecimal (uppercase).",                                 "FF"),
    ("synth-num-002",    "numeral_system",
     "Convert decimal 10 to binary.",                                                   "1010"),
    ("synth-num-003",    "numeral_system",
     "Convert decimal 8 to octal.",                                                     "10"),
    ("synth-grav-001",   "physics_gravity",
     "Object falls 2 s (g=9.8 m/s^2). Distance in m?",                                 "19.6"),
    ("synth-grav-002",   "physics_gravity",
     "Object falls 3 s (g=9.8 m/s^2). Distance in m?",                                 "44.1"),
    ("synth-eqnum-001",  "equation_numeric",
     "Solve for x: 2x + 4 = 10",                                                        "3"),
    ("synth-eqnum-002",  "equation_numeric",
     "Solve for x: 3x - 6 = 9",                                                         "5"),
    ("synth-eqnum-003",  "equation_numeric",
     "Solve for x: x/4 + 1 = 3",                                                        "8"),
    ("synth-eqsym-001",  "equation_symbolic",
     "Factor completely: x^2 - 4",                                                      "(x-2)(x+2)"),
    ("synth-eqsym-002",  "equation_symbolic",
     "Factor completely: x^2 - 9",                                                      "(x-3)(x+3)"),
]


def _make_val_row(eid, cat, prompt, gold):
    return SplitArtifactRow(
        example_id=eid, category=cat, prompt=prompt, gold=gold,
        source="synthetic-notebook-04", split="val",
        dataset_version=DATASET_VERSION, selection_seed=SEED,
        selection_rule="manual-coverage",
        selection_reason=f"val:manual-coverage; seed={SEED}",
    )


if VAL_ARTIFACT_PATH.exists():
    val_rows = read_split_jsonl(VAL_ARTIFACT_PATH)
    print(f"Loaded {len(val_rows)} val rows from {VAL_ARTIFACT_PATH.relative_to(REPO_ROOT)}")
else:
    val_rows = [_make_val_row(*r) for r in _SYNTHETIC_VAL]
    print(f"Using {len(val_rows)} synthetic val rows (val.jsonl not found - provisional).")

print(f"Categories ({len({r.category for r in val_rows})}): {sorted({r.category for r in val_rows})}")


In [ ]:
_SYNTHETIC_GOLDEN = [
    ("gold-bit-001",    "bit_manipulation",
     "XOR key='1010'. f('0101')=?",                       "1111"),
    ("gold-unit-001",   "unit_conversion",
     "Convert 3600 seconds to hours.",                         "1"),
    ("gold-cipher-001", "text_cipher",
     "Decode ROT-13: 'grfg'",                               "test"),
    ("gold-num-001",    "numeral_system",
     "Convert decimal 16 to binary.",                         "10000"),
    ("gold-grav-001",   "physics_gravity",
     "Object falls 1 s (g=9.8 m/s^2). Distance in m?",       "4.9"),
    ("gold-eqnum-001",  "equation_numeric",
     "Solve for x: x + 1 = 2",                               "1"),
    ("gold-eqsym-001",  "equation_symbolic",
     "Factor completely: x^2 - 1",                           "(x-1)(x+1)"),
]


def _make_golden_row(eid, cat, prompt, gold):
    return SplitArtifactRow(
        example_id=eid, category=cat, prompt=prompt, gold=gold,
        source="synthetic-notebook-04", split="golden",
        dataset_version=DATASET_VERSION, selection_seed=SEED,
        selection_rule="manual-coverage",
        selection_reason=f"golden:manual-coverage; seed={SEED}",
    )


if GOLDEN_ARTIFACT_PATH.exists():
    golden_rows = read_split_jsonl(GOLDEN_ARTIFACT_PATH)
    print(f"Loaded {len(golden_rows)} golden rows from {GOLDEN_ARTIFACT_PATH.relative_to(REPO_ROOT)}")
else:
    golden_rows = [_make_golden_row(*r) for r in _SYNTHETIC_GOLDEN]
    print(f"Using {len(golden_rows)} synthetic golden rows (golden.jsonl not found - provisional).")

print(f"Golden categories: {sorted({r.category for r in golden_rows})}")


In [ ]:
# ── Model + tokenizer load (Kaggle environment) ───────────────────────────────
import os, site, time
import torch
import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/usr/local/cuda/bin/ptxas"

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/"
site.addsitedir(cutlass_pkg_path)

import mamba_ssm  # noqa: must import before model load on Kaggle

MODEL_PATH = kagglehub.model_download(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    offload_folder="/kaggle/working/offload",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model.eval()
print("Model loaded successfully.")

## 4. Predictor (Real Model — Nemotron-3-Nano-30B)

The real predictor calls `metric/nemotron-3-nano-30b-a3b-bf16` via the Kaggle model hub. It applies the chat template, runs greedy decoding, and returns raw model output with actual latency and token counts.

**Baseline run:** `baseline-real-v1` — 200 examples, 0% accuracy.

**Known issue:** `max_new_tokens=512` is insufficient for this model's chain-of-thought. Nearly all predictions are truncated mid-reasoning trace and never reach a final answer. The one exception (`5a6ed2bf`, numeral_system) finished reasoning but the output parser failed to extract the answer from surrounding markdown.

**Two open problems handed off to `#21`:**
1. `max_new_tokens` needs to increase (2048–4096 range) to allow the model to complete reasoning.
2. The output parser contract is unresolved — `strip_v1` cannot extract a final answer from a completed reasoning trace. See `PENDING_REVIEW.md` output parser discrepancy.

The normalization, scoring, and reporting pipeline below is unchanged.

In [ ]:
# ── Real predictor (replaces stub) ───────────────────────────────────────────
import re

def _extract_boxed(text):
    """Pull answer from \\boxed{...} if present, else return last non-empty line."""
    match = re.search(r"\\boxed\{([^}]*)\}", text)
    if match:
        return match.group(1).strip()
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    return lines[-1] if lines else text.strip()


def real_predictor(req: PredictRequest) -> PredictResponse:
    messages = [
        {"role": "system", "content": "detailed thinking on"},
        {"role": "user",   "content": req.prompt},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    # apply_chat_template may return a BatchEncoding; unwrap if needed
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids.input_ids
    input_ids = input_ids.to(model.device)

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=DECODE_CONFIG["max_new_tokens"],
            do_sample=DECODE_CONFIG["do_sample"],
            temperature=DECODE_CONFIG["temperature"] if DECODE_CONFIG["do_sample"] else None,
        )
    latency_ms = (time.time() - t0) * 1000

    raw = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    tokens_in  = input_ids.shape[1]
    tokens_out = output_ids.shape[1] - input_ids.shape[1]

    return PredictResponse(
        prediction=raw,
        latency_ms=latency_ms,
        tokens_in=tokens_in,
        tokens_out=tokens_out,
    )


print("Real predictor ready.")
print(f"  Model device : {next(model.parameters()).device}")
print(f"  NORMALIZER_ID: {NORMALIZER_ID}  (normalization handled by pipeline, not predictor)")

## 5. Pipeline Execution

Running the full `ingest -> predict -> normalize -> score -> report` pipeline on the val split.

Artifacts written to `data/eval/runs/{RUN_ID}/`:

| Artifact | Contents |
|---|---|
| `predictions.jsonl` | Raw model output before normalization - allows re-scoring without re-running inference |
| `eval_records.jsonl` | Scored `EvalRecord` rows with full attribution |
| `run_config.json` | Snapshot of the run config |
| `summary.json` | Aggregate metrics derivable from records alone |

In [ ]:
OUTPUT_DIR = REPO_ROOT / "data" / "eval" / "runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

result = run_baseline_eval(
    split_rows=val_rows,
    config=run_config,
    predictor=real_predictor,
    output_dir=OUTPUT_DIR,
)

print(f"Pipeline complete - {len(result.records)} records")
print()
print("Artifacts:")
for name, path in result.artifact_paths.items():
    size = path.stat().st_size
    print(f"  {name:<18} {path.relative_to(REPO_ROOT)}  ({size} bytes)")


## 6. Evaluation Report

In [ ]:
s = result.summary

print("=" * 52)
print("  OVERALL ACCURACY")
print("=" * 52)
print(f"  Total examples : {s['total']}")
print(f"  Correct        : {s['correct']}")
print(f"  Accuracy       : {s['accuracy']:.1%}")
print(f"  Run ID         : {s['run_id']}")
print(f"  Model          : {s['model_id']}")
print(f"  Normalizer     : {s['normalizer_id']}")
print(f"  Seed           : {s['seed']}")
print()
lms = s["latency_ms"]
print(f"  Latency (ms)  : mean={lms['mean']:.1f}  p50={lms['p50']:.1f}  p95={lms['p95']:.1f}")
print(f"  Tokens in     : {s['tokens_in_total']} (total)")
print(f"  Tokens out    : {s['tokens_out_total']} (total)")


In [ ]:
print("=" * 62)
print("  PER-CATEGORY ACCURACY")
print("=" * 62)
print(f"  {'Category':<22} {'Correct':>8} {'Total':>7} {'Accuracy':>10}")
print(f"  {'-'*22} {'-'*8} {'-'*7} {'-'*10}")
for cat, st in sorted(s["per_category_accuracy"].items()):
    flag = " (high)" if st["accuracy"] >= 0.9 else (" (low)" if st["accuracy"] < 0.5 else "")
    print(
        f"  {cat:<22} {st['correct']:>8} {st['total']:>7}"
        f" {st['accuracy']:>9.1%}{flag}"
    )
print()
print("Legend: high >= 90%   low < 50%")


In [ ]:
failures = [r for r in result.records if not r.correct]

print(f"=== Failure Samples ({len(failures)} / {len(result.records)}) ===")
print()
for rec in failures:
    blank = rec.normalized_prediction == ""
    reason = "blank_prediction" if blank else "exact_match_fail"
    print(f"  example_id     : {rec.example_id}")
    print(f"  category       : {rec.category}")
    print(f"  gold           : {rec.gold!r}")
    print(f"  raw_prediction : {rec.prediction!r}")
    print(f"  normalized     : {rec.normalized_prediction!r}")
    print(f"  normalizer_id  : {rec.normalizer_id}")
    print(f"  failure_reason : {reason}")
    print()


In [ ]:
print("=== Normalizer Drift Analysis ===")
print("Failures that would pass under an alternative normalizer:")
print()

any_drift = False
for rec in failures:
    passing = {}
    for nid in available_normalizers():
        renorm = normalize(rec.prediction, normalizer_id=nid, category=rec.category)
        if renorm == rec.gold and nid != rec.normalizer_id:
            passing[nid] = renorm
    if passing:
        any_drift = True
        print(f"  {rec.example_id} ({rec.category})")
        print(f"    gold           = {rec.gold!r}")
        print(f"    raw_prediction = {rec.prediction!r}")
        print(f"    current ({rec.normalizer_id}): FAIL  normalized={rec.normalized_prediction!r}")
        for nid, renorm in sorted(passing.items()):
            print(f"    alt     ({nid}): PASS  normalized={renorm!r}")
        print()

if not any_drift:
    print("  No drift cases found - all failures are genuine wrong answers under all normalizers.")


## 7. Golden Regression Gate

The golden gate checks that the model answers every member of the frozen golden set correctly. A single miss blocks promotion.

**Policy (from `src/evaluation/golden_gate.py`):**
- Missing record for a golden example -> miss
- Incorrect `normalized_prediction` -> miss
- Nondeterministic results (same `example_id`, conflicting predictions) -> miss

In [ ]:
golden_run_config = make_run_config(
    run_id=RUN_ID + "-golden",
    model_id=MODEL_ID,
    prompt_template_id=PROMPT_TEMPLATE_ID,
    normalizer_id=NORMALIZER_ID,
    seed=SEED,
    split="golden",
    dataset_version=DATASET_VERSION,
    decode_config=DECODE_CONFIG,
)

GOLDEN_OUTPUT_DIR = REPO_ROOT / "data" / "eval" / "runs" / (RUN_ID + "-golden")

golden_result = run_baseline_eval(
    split_rows=golden_rows,
    config=golden_run_config,
    predictor=real_predictor,
    output_dir=GOLDEN_OUTPUT_DIR,
)

gate = evaluate_golden_gate(golden_result.records, golden_rows)
print(summarize_gate(gate))
print()
n_pass = sum(1 for r in golden_result.records if r.correct)
print(f"Correct: {n_pass} / {len(golden_result.records)}")
if gate.passed:
    print()
    print("Golden gate PASSED - stub baseline cleared the regression check.")
else:
    print()
    print("*** BLOCKED *** - investigate misses above before promoting this run.")


## 8. Artifact Round-Trip Validation

Verify that the artifacts written to disk reload correctly on a fresh kernel and that `summary.json` can be re-derived from `eval_records.jsonl` alone. This confirms the pipeline is stable across restarts.

In [ ]:
reloaded_records = read_eval_records_jsonl(OUTPUT_DIR / "eval_records.jsonl")
reloaded_summary = read_summary_json(OUTPUT_DIR / "summary.json")

assert len(reloaded_records) == len(result.records), (
    f"Record count mismatch: {len(reloaded_records)} != {len(result.records)}"
)

rederived = score_records(reloaded_records)
assert rederived["accuracy"] == reloaded_summary["accuracy"], (
    "Accuracy mismatch: summary.json out of sync with records"
)
assert rederived["correct"] == reloaded_summary["correct"], "Correct count mismatch"

for rec in reloaded_records:
    assert rec.run_id == RUN_ID
    assert rec.model_id == MODEL_ID
    assert rec.normalizer_id == NORMALIZER_ID
    assert rec.seed == SEED
    assert isinstance(rec.decode_config, dict)
    assert "temperature" in rec.decode_config
    assert "enable_thinking" in rec.decode_config

print("Round-trip validation PASSED.")
print(f"  Records reloaded  : {len(reloaded_records)}")
print(f"  Accuracy match    : {reloaded_summary['accuracy']:.1%}")
print(f"  Summary re-derive : OK")
print(f"  Attribution check : run_id, model_id, normalizer_id, seed, decode_config all present")


## Method and Outputs

### Method
1. **Ingest:** Load `SplitArtifactRow` records from `data/eval/val.jsonl` (or synthetic fallback).
2. **Predict:** Apply the real model predictor via `model.generate()` with parameters from `DECODE_CONFIG`.
3. **Normalize:** Apply the versioned normalizer specified by `NORMALIZER_ID`.
4. **Score:** Exact-match `normalized_prediction` against `gold`.
5. **Report:** Write four artifacts; display overall + per-category accuracy; flag failure cases and normalizer drift.

### Outputs (baseline-real-v1)
- `docs/eval/baseline_summary.json` - committed; aggregate metrics for this run
- `baseline_results.jsonl` - (200 raw predictions, lives in `/kaggle/working/`)

## Provisional Assumptions

| # | Assumption | Status | Risk if wrong | Mitigation |
|---|---|---|---|---|
| A1 | Competition evaluator uses exact-match (possibly with strip) | Unconfirmed | Local accuracy over-counts real score | Run dual-normalizer eval (`exact_v1` + `strip_v1`); report both |
| A2 | `strip_v1` is the right baseline normalizer | **Invalidated** — model output is a reasoning trace, not a plain answer | Normalizer can never match gold on truncated output | Blocked on `#21` output parser resolution |
| A3 | Val + golden splits are real Kaggle data | Confirmed — 200 real examples used in `baseline-real-v1` | — | — |
| A4 | Stub predictor accuracy (~68%) does not represent real baseline | **Confirmed** — real baseline is 0% | — | Real baseline run complete; root cause identified |
| A5 | Decode: greedy, `enable_thinking=False` | Confirmed for this run — model still produces long reasoning traces regardless | Reasoning trace fills context before final answer is written | `max_new_tokens` must increase to 2048–4096; issue `#21` sweeps this |
| A6 | Token counts use `len(prompt)//4` proxy | **Fixed** — real run uses `tokenizer.encode()` counts | — | — |

## Results / Open Risks

**Current state:** The pipeline is validated end-to-end with the real Nemotron-3-Nano-30B model. All four artifact types write and reload correctly. Baseline run `baseline-real-v1` completed 200 examples with **0% accuracy** — root cause identified as `max_new_tokens=512` truncating the model's reasoning trace before it reaches a final answer.

**Open risks:**
- `max_new_tokens=512` must be increased to 2048–4096 before any meaningful accuracy signal is possible. Handed off to `#21`.
- The output parser contract is unresolved. Even when the model completes reasoning (one case observed), `strip_v1` cannot extract the answer from surrounding markdown or prose. `#21` must resolve this before sweep results can be trusted.
- The golden gate will fail under the real model until the token limit and parser issues are resolved. This is expected and does not indicate a pipeline bug.
- Category-specific normalization (e.g., case folding for `numeral_system`) may be needed — deferred to `#21`.

## Sources

- [`docs/architecture/ARCHITECTURE.md`](../docs/architecture/ARCHITECTURE.md) - EvalRecord contract and evaluation layer design
- [`docs/architecture/COMPETITION.md`](../docs/architecture/COMPETITION.md) - answer format and scoring assumptions
- [`docs/analysis/ADVERSARIAL_REVIEW.md`](../docs/analysis/ADVERSARIAL_REVIEW.md) - exact-match rationale vs. LaTeX formatting
- [`docs/planning/plan_v0.2.md`](../docs/planning/plan_v0.2.md) - evaluation protocol and determinism requirements
- [`docs/execution/plans/issue-19-baseline-eval-and-normalization.md`](../docs/execution/plans/issue-19-baseline-eval-and-normalization.md) - task breakdown
- [`src/evaluation/normalization.py`](../src/evaluation/normalization.py) - versioned normalizer registry
- [`src/evaluation/runner.py`](../src/evaluation/runner.py) - pipeline implementation
- [`src/evaluation/golden_gate.py`](../src/evaluation/golden_gate.py) - regression gate implementation
- [`src/contracts.py`](../src/contracts.py) - canonical data contracts (EvalRecord, ReasoningExample)